In [6]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [12]:
import numpy as np

scenario = 1 # 0 = A, 1 = B, 2 = C

In [13]:

def build_matrix(scenario):
    """
    Build the 76x76 transition matrix for the Smart Home Energy System.

    State encoding:  index = B*15 + W*3 + C   (0..74 = operational, 75 = failure)
      B in {0,1,2,3,4}  – battery level
      W in {0,1,2,3,4}  – weather  (0=Stormy … 4=Sunny)
      C in {0,1,2}      – consumption (0=Low … 2=High)

    Physics:
      B' = clip(B + W - C, 0, 4)   (deterministic given current state)

    Stochastic environment:
      Weather : P(stay)=0.6, P(±1)=0.2 each  (reflective boundaries)
      Consumption: P(stay)=0.7, P(±1)=0.15 each (reflective boundaries)
    """

    P = np.zeros((76, 76))

    # ── helper: build marginal transition distributions ─────────────────────
    def marginal(state, max_val, p_stay, p_step):
        """Return dict {next_state: probability} with reflective boundaries."""
        dist = {state: p_stay}
        for delta in (-1, +1):
            nb = state + delta
            if 0 <= nb <= max_val:
                dist[nb] = dist.get(nb, 0) + p_step
            else:                          # reflect probability back to boundary
                dist[state] = dist.get(state, 0) + p_step
        return dist

    # ── fill operational rows (0..74) ────────────────────────────────────────
    for i in range(75):
        B = i // 15
        W = (i % 15) // 3
        C = i % 3

        B_prime = int(np.clip(B + W - C, 0, 4))

        w_dist = marginal(W, 4, p_stay=0.6, p_step=0.2)
        c_dist = marginal(C, 2, p_stay=0.7, p_step=0.15)

        for w_next, pw in w_dist.items():
            for c_next, pc in c_dist.items():
                j = B_prime * 15 + w_next * 3 + c_next
                P[i, j] += pw * pc

    # ── failure state is a sink ──────────────────────────────────────────────
    P[75, 75] = 1.0

    # ── scenario modifications ───────────────────────────────────────────────
    if scenario == 1:  # Scenario B – Brittle-Bottom
        # If P(row → 0) > 10 %, redirect exactly 10 % to failure
        for row in range(75):
            if P[row, 0] > 0.10:
                P[row, 75] += 0.10
                P[row, 0]  -= 0.10

    elif scenario == 2:  # Scenario C – Extreme-Stress
        # For every next operational state where B'=0 or B'=4 and P>5 %,
        # redirect exactly 5 % to failure
        for row in range(75):
            for j in range(75):
                b_prime = j // 15
                if (b_prime == 0 or b_prime == 4) and P[row, j] > 0.05:
                    P[row, 75] += 0.05
                    P[row, j]  -= 0.05

    return P


M = build_matrix(scenario)
print("Shape     :", M.shape)
print(M)


Shape     : (76, 76)
[[0.58 0.12 0.   ... 0.   0.   0.1 ]
 [0.02 0.56 0.12 ... 0.   0.   0.1 ]
 [0.   0.12 0.68 ... 0.   0.   0.  ]
 ...
 [0.   0.   0.   ... 0.56 0.12 0.  ]
 [0.   0.   0.   ... 0.12 0.68 0.  ]
 [0.   0.   0.   ... 0.   0.   1.  ]]


In [15]:

# ── Initial state s0 = (B=2, W=2, C=1) ─────────────────────────────────────
s0 = 2 * 15 + 2 * 3 + 1   # = 37

def distribution_after_n_steps(P, s0, N):
    """
    Compute the distribution µ_N after N steps via power iteration.

    µ_0 is a point mass on s0.
    µ_{n+1} = µ_n @ P   (row vector × transition matrix)

    Parameters
    ----------
    P  : (76, 76) row-stochastic transition matrix
    s0 : int, index of the initial state
    N  : int, number of steps

    Returns
    -------
    mu : (76,) array, probability distribution over states after N steps
    """
    mu = np.zeros(P.shape[0])
    mu[s0] = 1.0
    for _ in range(N):
        mu = mu @ P
    return mu


def stationary_distribution(P, tol=1e-10):
    """
    Compute the stationary distribution π via eigen-decomposition.

    π satisfies  π @ P = π  (left eigenvector for eigenvalue 1).
    We find it as the left eigenvector of P corresponding to eigenvalue 1,
    which equals the right eigenvector of P^T for eigenvalue 1.

    If the chain has an absorbing state (e.g. failure state 75) the unique
    stationary distribution concentrates entirely on that absorbing class.

    Parameters
    ----------
    P   : (76, 76) row-stochastic transition matrix
    tol : tolerance for identifying eigenvalue = 1

    Returns
    -------
    pi : (76,) array, stationary distribution (normalised, non-negative),
         or None if no eigenvalue-1 eigenvector with non-negative entries
         is found.
    """
    eigenvalues, eigenvectors = np.linalg.eig(P.T)   # right eigs of P^T = left eigs of P

    # find all eigenvalues close to 1
    candidates = []
    for idx, ev in enumerate(eigenvalues):
        if np.isclose(ev.real, 1.0, atol=tol) and np.isclose(ev.imag, 0.0, atol=tol):
            vec = eigenvectors[:, idx].real
            if np.all(vec >= -tol):               # must be non-negative
                vec = np.maximum(vec, 0)
                vec /= vec.sum()                  # normalise to a probability vector
                candidates.append(vec)

    if not candidates:
        print("No stationary distribution found.")
        return None

    if len(candidates) > 1:
        print(f"Multiple ({len(candidates)}) stationary distributions found "
              "(chain has several closed communicating classes).")

    return candidates[0] if len(candidates) == 1 else candidates


# ── Demo ─────────────────────────────────────────────────────────────────────
for N in [1, 10, 100]:
    mu = distribution_after_n_steps(M, s0, N)
    print(f"µ_{N:>3d}  sum={mu.sum():.6f}  "
          f"top-3 states: {np.argsort(mu)[-3:][::-1].tolist()}  "
          f"P(failure)={mu[75]:.6f}")

pi = stationary_distribution(M)
if pi is not None:
    print(f"\nStationary π  sum={pi.sum():.6f}  "
          f"top-3 states: {np.argsort(pi)[-3:][::-1].tolist()}  "
          f"π(failure)={pi[75]:.6f}")


µ_  1  sum=1.000000  top-3 states: [52, 55, 49]  P(failure)=0.000000
µ_ 10  sum=1.000000  top-3 states: [66, 69, 70]  P(failure)=0.014376
µ_100  sum=1.000000  top-3 states: [75, 72, 73]  P(failure)=0.327109

Stationary π  sum=1.000000  top-3 states: [75, 74, 73]  π(failure)=1.000000
